# 02 - Recopilacion de Datos

**Objetivo**: Recopilar y cachear datos crudos de todas las APIs.

Este notebook ejecuta el pipeline de recopilacion:
1. Descubrir pools nuevos en Solana, Ethereum y Base
2. Enriquecer con datos de DexScreener
3. Obtener OHLCV historico
4. Obtener holders (Solana)
5. Verificar contratos (ETH/Base)
6. Obtener contexto de mercado (BTC/ETH/SOL)

> **Nota**: La primera ejecucion tomara varios minutos por los rate limits.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.collector import DataCollector
from src.data.storage import Storage
import config

print("Modulos importados correctamente")

In [ ]:
# Crear el collector y storage
storage = Storage()
collector = DataCollector(storage=storage)

print(f"Base de datos: {config.DB_PATH}")
print(f"Estado actual de la BD:")
for table, count in storage.stats().items():
    print(f"  {table}: {count} registros")

## Paso 1: Descubrir Pools Nuevos

Buscamos los pools creados en las ultimas 48 horas en cada cadena.
Esto es la base de nuestra **recopilacion prospectiva** (capturar tokens desde su nacimiento).

In [ ]:
# Descubrir pools nuevos en todas las cadenas
# pages=3 obtiene ~90 pools por cadena (30 por pagina)
new_tokens = collector.discover_new_pools(
    chains=["solana", "ethereum", "base"],
    pages=3,
)

print(f"\nTotal tokens descubiertos: {len(new_tokens)}")

# Mostrar distribucion por cadena
from collections import Counter
chain_counts = Counter(t.get('chain') for t in new_tokens)
for chain, count in chain_counts.items():
    print(f"  {chain}: {count} tokens")

## Paso 2: Enriquecer con DexScreener

Para cada token, obtenemos buyers/sellers counts de DexScreener.
Esta es la **unica fuente gratuita** para estos datos.

In [ ]:
# Enriquecer con datos de DexScreener (buyers/sellers)
# Esto puede tardar unos minutos
collector.enrich_with_dexscreener(new_tokens)

## Paso 3: Obtener OHLCV Historico

Datos de velas (Open, High, Low, Close, Volume) para calcular features de price action.

In [ ]:
# Obtener OHLCV diario (ultimos 30 dias)
collector.collect_ohlcv(new_tokens, timeframe="day", limit=30)

# Tambien obtener OHLCV horario para las primeras 48 horas
collector.collect_ohlcv(new_tokens, timeframe="hour", limit=48)

## Paso 4: Obtener Holders (Solana)

Top 20 holders para calcular concentracion. Solo funciona para tokens de Solana.

In [ ]:
# Obtener holders de tokens Solana
if config.HELIUS_API_KEY:
    collector.collect_holders(new_tokens)
else:
    print("Saltando holders: HELIUS_API_KEY no configurada")

## Paso 5: Verificar Contratos

Verificar si el source code esta publicado (solo ETH/Base con Etherscan).

In [ ]:
# Verificar contratos (ETH/Base)
if config.ETHERSCAN_API_KEY:
    collector.collect_contract_info(new_tokens)
else:
    print("Saltando verificacion de contratos: ETHERSCAN_API_KEY no configurada")

## Paso 6: Contexto de Mercado

Precios historicos de BTC, ETH y SOL para features de contexto.

In [ ]:
# Obtener precio historico de BTC, ETH, SOL
collector.collect_market_context(days=90)
print("Datos de mercado guardados en data/processed/")

## Resumen de la Recopilacion

In [ ]:
# Estado final de la base de datos
print("=" * 50)
print("ESTADO DE LA BASE DE DATOS")
print("=" * 50)
for table, count in storage.stats().items():
    print(f"  {table}: {count} registros")

# Tokens por cadena
tokens_df = storage.get_all_tokens()
if not tokens_df.empty:
    print(f"\nTokens por cadena:")
    print(tokens_df['chain'].value_counts().to_string())

### Siguiente paso
Ejecutar `03_data_cleaning.ipynb` para limpiar y normalizar los datos recopilados.